# 三维道路 DAS + 锤击地下空洞探测算法原型

## 当前项目状态摘要

当前阶段：**Stage 2B**。

当前默认几何：道路 `80 m x 15 m`，DAS 光纤位于 `y=0 m`，锤击炮线位于 `y=15 m`，空洞中心为 `void_xyz=(40.0, 7.5, 2.0)`，`depth` 向下为正。

当前可用流程：

1. `kinematic`：默认稳定后端，三维运动学点绕射正演 + 三维 `x-y-depth` 定位闭环；
2. `devito_acoustic_3d`：代码接口和 acoustic 方程路径已实现，`myvoid` 中 Devito 已安装，但当前 Windows 原生 runtime 还没有通过极小 Operator smoke test；
3. `OpenSWPCBackend`：仍为外部程序占位，当前未编译未配置。

当前 DAS：仍是光纤 polyline 采样点接收器近似，不是真实 gauge-length averaged axial strain。

当前定位：用于验证三维几何、数据结构和流程贯通，暂不以误差最小为核心指标。


## 1. 三维坐标系统

本项目始终是三维道路场景，不是二维 `x-depth` 剖面包装：

- `x`：沿道路或 DAS 光纤方向；
- `y`：横穿道路方向；
- `depth`：向下为正；
- `source_xyz`：锤击点三维坐标；
- `receiver_polyline`：DAS 光纤三维折线；
- `receiver_xyz`：沿光纤采样得到的三维接收点；
- `void_xyz` / `VoidBody3D.center_xyz`：空洞或低速异常体的三维位置。


## 2. 默认三维道路、光纤、锤击点和空洞几何

当前默认场景表达的是道路两侧观测：DAS 光纤沿 `y=0 m` 一侧布设，锤击点沿 `y=15 m` 另一侧布设，空洞位于道路中部浅层地下。

这类几何天然会引入横向 `y` 与深度 `depth` 的分辨率问题，因此后续不能只看 `x-depth` 切片。


In [ ]:
from main import build_stage1_scene

scene = build_stage1_scene()
print("road_length_m =", scene["road_length_m"])
print("road_width_m =", scene["road_width_m"])
print("fiber_y_m =", scene["fiber_y_m"])
print("source_y_m =", scene["source_y_m"])
print("void_xyz =", scene["void_model"].void_xyz.xyz)
print("velocity_grid_shape =", scene["velocity_grid"].shape)


## 3. 三维运动学绕射正演原理

默认 `kinematic` 后端仍使用三维运动学点绕射公式：

```text
总走时 = 震源到空洞的走时 + 空洞到接收点的走时
```

即：

```text
t_total = |source_xyz - void_xyz| / v + |void_xyz - receiver_xyz| / v
```

这个后端的价值是验证三维几何、炮集维度和定位目标函数是否贯通。它不是声波或弹性波方程求解器。


## 4. Stage 2B：为什么优先接入 Devito

Devito 的优势是 Python 接入轻、3D acoustic 示例完整、可以把 `VelocityGrid3D` 直接转为规则网格，并输出接收记录和波场快照。它适合作为本项目从运动学正演走向真实波动方程正演的第一步。

但 Devito acoustic 仍然只是标量声波近似，不是弹性波。它不能直接给 DAS 轴向应变；真实 DAS 还需要弹性位移场、应变张量和沿光纤切向量的 gauge-length 处理。


## 5. Devito 安装与运行状态

当前 `myvoid` 中已安装：

```text
devito 4.8.22
m2w64-toolchain 5.3.0
```

当前状态必须分开理解：

- Devito 可导入：是；
- Devito 极小 Operator runtime：否；
- 当前阻塞：Windows 原生 Python / Devito / CodePy / MinGW 的 JIT 路径和 POSIX 假设问题。

因此本项目现在不会伪造 Devito 炮集或波场快照。


In [ ]:
from hcz_road_void.forward import DevitoBackend

status = DevitoBackend().runtime_status()
print(status.as_dict())


## 6. Devito acoustic 与 kinematic 的区别

`kinematic` 后端只计算到时并在到时处放置 Ricker 子波；它没有网格波场变量。

`devito_acoustic_3d` 后端的目标是在 `VelocityGrid3D` 上求解：

```text
m * u.dt2 - laplace(u) = source
m = 1 / vp^2
```

一旦 runtime 可用，Devito 后端将输出：

1. `ForwardResult3D.data`：`n_sources x n_receivers x n_times`；
2. `devito_synthetic_gather.png`：真实 acoustic 炮集；
3. `devito_wavefield_snapshots/`：标量声波场快照；
4. `devito_wavefield_animation.gif`：标量声波场动图。


## 7. VelocityGrid3D、VoidBody3D 与 Devito 的衔接

Stage 2A 已经把异常体从单点推进到 `VoidBody3D`，并能把低速体嵌入 `VelocityGrid3D`。Stage 2B 的 Devito 后端优先读取这个速度网格，而不是在后端里硬编码另一套空洞位置。

这保证了 main.py、可视化、正演后端和后续定位共享同一组三维参数。


In [ ]:
from hcz_road_void.forward import velocity_grid_to_devito_inputs

converted = velocity_grid_to_devito_inputs(scene["velocity_grid"])
print("shape =", converted["shape"])
print("origin =", converted["origin"])
print("extent =", converted["extent"])
print("spacing =", converted["spacing"])
print("axis_order =", converted["axis_order"])


## 8. 当前 main.py 输出状态

默认运行：

```powershell
D:\HczApp\Anaconda\envs\myvoid\python.exe main.py
```

仍使用 `kinematic`，会输出几何图、速度模型切片、合成炮集、三维定位目标函数切片和 `run_summary.json`。

显式运行 Devito：

```powershell
D:\HczApp\Anaconda\envs\myvoid\python.exe main.py --backend devito_acoustic_3d
```

当前 Windows 原生环境会给出中文错误，说明 Devito 已安装但 runtime 未通过，不会生成伪 Devito 输出。


## 9. DAS 光纤观测的当前近似

当前 `receiver_polyline` 已经能按固定间距采样为 `receiver_xyz`，并保存 gauge length metadata。但当前合成记录仍是点接收器近似，不是真实 DAS 轴向应变。

真正 DAS 需要：

1. 弹性位移场；
2. 应变张量；
3. 光纤局部切向量；
4. gauge length 平均；
5. 可能还需要应变率或相位响应。

Devito acoustic 不能直接完成这个步骤。


## 10. 定位不确定性和当前不追求误差最小

当前定位模块的任务是验证三维观测系统和数据结构是否贯通，不是把 `best_xyz` 对单个合成例子调到最接近 `true_xyz`。

只有当正演从运动学近似推进到更真实的三维声波/弹性波模型之后，才应该系统优化定位目标函数、速度误差处理和不确定性分析。


## 11. 当前限制

1. 默认结果仍来自三维运动学点绕射近似；
2. Devito 已安装但当前 Windows 原生 runtime 未通过；
3. Devito acoustic 不是弹性波；
4. 当前没有 PML、自由表面、真实空洞边界散射和 DAS 轴向应变；
5. OpenSWPC 未编译未接入；
6. 当前不把定位误差当成核心验收指标。


## 12. 下一步计划

1. 优先在 WSL/Linux 或 Devito 官方 Docker 中验证当前 `DevitoBackend`；
2. 如果必须走 Windows 原生路线，继续定位 CodePy/MinGW 路径转义和 allocator 问题；
3. Devito acoustic runtime 通过后，生成真实炮集、波场快照和 GIF；
4. 再考虑把 Devito 输出接入定位模块做对比；
5. 后续转向 Devito elastic 或 OpenSWPC，推进真实 DAS 轴向应变。
